# Phase 3 · Apache Spark — tầng 3 của lakehouse

Phase 1 và 2 đào **sâu xuống** tầng dưới. Phase 3 dựng hẳn một tầng mới **lên trên**:
compute engine. Databricks gọi tầng này là *Databricks Runtime*.

```
jupyter ──sc://15002──► spark-connect (DRIVER) ──7077──► spark-master
  (notebook này)              │                              │
                          UI :4040                    ┌──────┴──────┐
                                                  worker-1      worker-2
                                                  2 core/4GB    2 core/4GB
```

**Điều dễ hiểu nhầm nhất, nói ngay từ đầu:** notebook này **không chứa Spark**.
Nó không có JVM, không có driver, không tính toán gì cả. Nó gửi *mô tả phép tính*
qua gRPC tới container `spark-connect`, và nhận về *kết quả*. Đúng mô hình
**Databricks Connect**.

Mở sẵn hai tab trước khi bắt đầu — cả phase này sẽ đọc chúng liên tục:

| | |
|---|---|
| **http://localhost:8080** | Master UI — worker nào còn sống |
| **http://localhost:4040** | Spark UI — job, stage, shuffle. **Quan trọng nhất.** |

> **Cách học:** chạy cell → **nhìn số của mình** → mở UI đối chiếu → rồi mới đọc giải thích.
> Đọc trước thì bạn đang học thuộc, không phải đang hiểu.

## Chuẩn bị · Biến, tiện ích, và đưa dữ liệu lên MinIO

In [3]:
import os, json, time, pathlib, urllib.request
import boto3
from pyspark.sql import SparkSession, functions as F

BUCKET = os.environ['LAKEHOUSE_BUCKET']
REMOTE = os.environ['SPARK_REMOTE']

# CHÚ Ý tiền tố: s3a:// chứ không phải s3://
#   s3://  là cách DuckDB và delta-rs gọi (client S3 riêng của chúng)
#   s3a:// là cài đặt S3 của Hadoop — thứ Spark dùng
# Cùng một object trên MinIO, chỉ khác thư viện nào đang đọc.
RAW   = f's3a://{BUCKET}/raw/yellow_taxi/'
ZONES = f's3a://{BUCKET}/raw/taxi_zone_lookup.csv'
DELTA_PHASE2 = f's3a://{BUCKET}/phase2/bronze_trips'   # bảng delta-rs đã tạo ở Phase 2
GOLD  = f's3a://{BUCKET}/phase3/gold_borough'

print('Spark Connect :', REMOTE)
print('Dữ liệu thô   :', RAW)

Spark Connect : sc://spark-connect:15002
Dữ liệu thô   : s3a://lakehouse/raw/yellow_taxi/


In [4]:
# Đẩy toàn bộ file trong data/ lên MinIO. Chạy lại nhiều lần vô hại — có rồi thì bỏ qua.
s3 = boto3.client('s3')

def da_co(prefix):
    r = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    return {o['Key']: o['Size'] for o in r.get('Contents', [])}

co = da_co('raw/')
for f in sorted(pathlib.Path('/home/jovyan/data').glob('*')):
    if f.suffix not in ('.parquet', '.csv'):
        continue
    key = f'raw/yellow_taxi/{f.name}' if f.suffix == '.parquet' else f'raw/{f.name}'
    if co.get(key) == f.stat().st_size:
        continue
    s3.upload_file(str(f), BUCKET, key)
    print(f'↑ {key}  ({f.stat().st_size/1e6:.0f} MB)')

tong = da_co('raw/yellow_taxi/')
print(f'\n✓ {len(tong)} file thô trên MinIO, tổng {sum(tong.values())/1e6:.0f} MB')


✓ 12 file thô trên MinIO, tổng 693 MB


In [5]:
# ── Hai tiện ích dùng suốt notebook ──────────────────────────────

def api(path, host='spark-connect:4040'):
    '''Đọc thẳng REST API của Spark UI.

    Nguyên tắc 'đọc file thô' của lộ trình, áp lên chính Spark: mọi con số
    bạn thấy trên giao diện :4040 đều đến từ đây. Đọc API thay vì nhìn ảnh
    nghĩa là bạn có thể *đo* thay vì *ước lượng*.
    '''
    with urllib.request.urlopen(f'http://{host}{path}', timeout=15) as r:
        return json.load(r)

def bench(nhan, fn):
    '''Bấm giờ một action. Trả về (giây, kết quả).'''
    t0 = time.perf_counter()
    kq = fn()
    dt = time.perf_counter() - t0
    print(f'{nhan:<44} {dt:7.2f}s')
    return dt, kq

---
## Bước 1 · Nối vào cluster, và tìm xem mình đang đứng ở đâu

`SparkSession.builder.remote(...)` chứ không phải `.master(...)`. Một chữ khác nhau
nhưng là hai thế giới: `.master()` khởi động một Spark **bên trong tiến trình này**,
`.remote()` nối tới một Spark **đã chạy sẵn ở nơi khác**.

In [6]:
spark = SparkSession.builder.remote(REMOTE).getOrCreate()
print('Spark version:', spark.version)

Spark version: 4.1.3


In [10]:
import socket
APP = api('/api/v1/applications')[0]['id']

print('Notebook này chạy trên container :', socket.gethostname())
print('Application id trên cluster      :', APP)
print()

print('── Executor mà cluster đang cấp cho ta ' + '─'*30)
for e in api(f'/api/v1/applications/{APP}/executors'):
    vai = 'DRIVER  ' if e['id'] == 'driver' else f"executor {e['id']}"
    print(f"  {vai}  {e['hostPort']:<26} {e['totalCores']} core   {e['maxMemory']/2**20:6.0f} MB")

Notebook này chạy trên container : 3699fecf0ffe
Application id trên cluster      : app-20260823140503-0000

── Executor mà cluster đang cấp cho ta ──────────────────────────────
  DRIVER    spark-connect:40905        0 core     1049 MB
  executor 1  172.18.0.4:37297           2 core     1663 MB
  executor 0  172.18.0.5:41273           2 core     1663 MB


In [11]:
# Nhìn từ phía Master: nó biết những worker nào?
m = api('/json/', host='spark-master:8080')
print(f"Master: {m['status']}  —  {m['aliveworkers']} worker ALIVE, "
      f"tổng {m['cores']} core / {m['memory']/1024:.0f} GB\n")
for w in m['workers']:
    print(f"  {w['id']:<42} {w['state']:<6} {w['cores']} core  {w['memory']/1024:.0f} GB")

Master: ALIVE  —  2 worker ALIVE, tổng 4 core / 6 GB

  worker-20260823140502-172.18.0.5-42369     ALIVE  2 core  3 GB
  worker-20260823140502-172.18.0.4-32811     ALIVE  2 core  3 GB


### Đọc kỹ ba con số vừa in

**`driver` nằm ở `spark-connect`, không phải ở đây.** Container notebook không xuất hiện
trong danh sách trên. Nó là client, không phải thành viên cluster.

**Driver có core nhưng không có việc.** Driver không tính toán — nó lập kế hoạch, chia
việc, và **giữ kết quả `collect()` trong RAM của nó**. Đây là lý do `df.collect()` trên
bảng 100GB làm sập cả ứng dụng, dù cluster có 500 core: 100GB đó phải chui hết vào một JVM.

**Bốn core executor.** Nghĩa là tối đa **4 task chạy song song**. Con số này quyết định
mọi thứ ở các bước sau — hãy nhớ nó.

> Đối chiếu ngay: mở http://localhost:8080, bạn phải thấy đúng 2 worker `ALIVE` và một
> application đang `RUNNING`. Bấm vào tên application sẽ nhảy sang :4040.

---
## Bước 2 · Lazy evaluation — tự chứng minh, đừng tin lời sách

Sách nào cũng viết "Spark lazy". Ta không tin, ta đếm.

`/api/v1/applications/{app}/jobs` cho biết cluster đã chạy bao nhiêu **job**.
Một job = một lần Spark thật sự đụng vào dữ liệu.

In [12]:
def so_job():
    return len(api(f'/api/v1/applications/{APP}/jobs'))

truoc = so_job()

# Ba dòng "biến đổi" trên dữ liệu. Chạy tức thì.
t0 = time.perf_counter()
df   = spark.read.parquet(RAW)
loc  = df.filter(F.col('trip_distance') > 1)
gon  = loc.select('tpep_pickup_datetime', 'PULocationID', 'trip_distance', 'total_amount')
print(f'Ba lệnh transformation mất : {time.perf_counter()-t0:.3f}s')
print(f'Số job cluster đã chạy      : {truoc} → {so_job()}')

Ba lệnh transformation mất : 0.001s
Số job cluster đã chạy      : 0 → 0


In [13]:
truoc = so_job()

# Action ĐẦU TIÊN còn phải trả một khoản phí một lần: liệt kê file trên MinIO
# và đọc footer của từng file. Trả trước ở đây để phép đo bên dưới sạch.
_ = df.select('PULocationID').limit(1).collect()

t0 = time.perf_counter()
n = df.count()               # ← không lọc gì cả
print(f'df.count()          = {n:,} dòng, mất {time.perf_counter()-t0:.2f}s')

t0 = time.perf_counter()
tb = gon.agg(F.avg('total_amount')).collect()[0][0]
print(f'avg(total_amount)   = {tb:.2f},        mất {time.perf_counter()-t0:.2f}s')

print(f'\nSố job cluster đã chạy: {truoc} → {so_job()}')

df.count()          = 41,169,720 dòng, mất 0.52s
avg(total_amount)   = 31.92,        mất 0.89s

Số job cluster đã chạy: 0 → 6


### Chuyện gì vừa xảy ra

`read` + `filter` + `select` không sinh job nào. Chỉ action mới sinh.

Và hai action không hề bằng nhau. `df.count()` gần như tức thì vì **Parquet ghi sẵn số dòng
trong footer** — Spark chỉ việc cộng metadata của 12 file, gần như không đọc dữ liệu. Đúng
thứ bạn đã mổ bằng tay ở Phase 1 bước 3.

Còn `avg(total_amount)` thì buộc phải đọc thật cả một cột của 31 triệu dòng.

> **Chi tiết dễ đo nhầm:** action đầu tiên trong phiên luôn đắt bất thường, vì nó còn phải
> liệt kê file trên object storage và đọc footer. Cell trên gọi `limit(1).collect()` để
> trả trước khoản đó — nếu không, bạn sẽ đo nhầm phí khởi động thành phí tính toán.

In [14]:
gon.explain(mode='formatted')

== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [tpep_pickup_datetime#2, trip_distance#5, PULocationID#8, total_amount#17]
Batched: true
Location: InMemoryFileIndex [s3a://lakehouse/raw/yellow_taxi]
PushedFilters: [IsNotNull(trip_distance), GreaterThan(trip_distance,1.0)]
ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz,trip_distance:double,PULocationID:int,total_amount:double>

(2) ColumnarToRow [codegen id : 1]
Input [4]: [tpep_pickup_datetime#2, trip_distance#5, PULocationID#8, total_amount#17]

(3) Filter [codegen id : 1]
Input [4]: [tpep_pickup_datetime#2, trip_distance#5, PULocationID#8, total_amount#17]
Condition : (isnotnull(trip_distance#5) AND (trip_distance#5 > 1.0))

(4) Project [codegen id : 1]
Output [4]: [tpep_pickup_datetime#2, PULocationID#8, trip_distance#5, total_amount#17]
Input [4]: [tpep_pickup_datetime#2, trip_distance#5, PULocationID#8, total_amount#17]




Tìm hai dòng trong plan trên:

- **`PushedFilters: [..., GreaterThan(trip_distance,1.0)]`** — `filter` được đẩy xuống
  tầng đọc Parquet. Chính là **predicate pushdown** bạn đo bằng tay ở Phase 1, nay Spark
  tự làm.
- **`ReadSchema: struct<...>`** — chỉ liệt kê 4 cột. Mười lăm cột còn lại không bao giờ
  rời khỏi đĩa. Đó là **projection pushdown**.

Nếu Spark chạy ngay từng dòng lệnh, cả hai tối ưu này đều bất khả thi. **Lười không phải
là nhược điểm, nó là điều kiện để tối ưu.**

---
## Bước 3 · Partition sinh ra từ đâu

**Partition** là đơn vị song song của Spark: một partition ↔ một task ↔ một core tại một
thời điểm. Nhưng con số partition ở đâu ra? Không phải Spark bốc đại.

In [15]:
# spark_partition_id() trả về partition của từng dòng. Đếm giá trị phân biệt = số partition.
so_partition = df.select(F.spark_partition_id().alias('p')).distinct().count()

files = da_co('raw/yellow_taxi/')
print(f'Số file Parquet trên MinIO : {len(files)}')
print(f'Tổng dung lượng            : {sum(files.values())/1e6:.0f} MB')
print(f'Số partition Spark tạo ra  : {so_partition}')
print(f'Số core executor có sẵn    : 4')

Số file Parquet trên MinIO : 12
Tổng dung lượng            : 693 MB
Số partition Spark tạo ra  : 6
Số core executor có sẵn    : 4


### Ba con số này quan hệ thế nào

Spark cắt dữ liệu theo `spark.sql.files.maxPartitionBytes` (mặc định **128MB**), nhưng
**không cắt ngang row group** — nhớ lại cấu trúc Parquet bạn đã mổ ở Phase 1 bước 3.
Nên số partition ≈ tổng dung lượng ÷ 128MB, làm tròn theo ranh giới file và row group.

Hai trạng thái cần tránh, và cả hai đều đã gặp ở Phase 1:

- **Quá ít partition** → có core ngồi không. 4 core mà 2 partition thì một nửa cluster rảnh.
- **Quá nhiều partition** → mỗi task chỉ làm vài mili-giây, thời gian *điều phối* task
  vượt thời gian *làm việc*. Đây chính là **small files problem** của Phase 1 bước 6,
  nhìn từ phía compute thay vì phía storage.

Quy tắc thô: nhắm **2–4 partition cho mỗi core**, mỗi partition cỡ 100–200MB.

---
## Bước 4 · Narrow vs wide — ý niệm quan trọng nhất của cả phase ⭐

Mọi câu hỏi về hiệu năng Spark, cuối cùng đều quy về một câu: **có shuffle không, và bao nhiêu?**

- **Narrow** — mỗi partition đầu ra chỉ cần **một** partition đầu vào. `filter`, `select`,
  `withColumn`. Dữ liệu **không rời khỏi máy**.
- **Wide** — một partition đầu ra cần **nhiều** partition đầu vào. `groupBy`, `join`,
  `distinct`, `orderBy`. Dữ liệu phải **đi qua network** để những dòng cùng khoá gặp nhau.

Cuộc di chuyển đó gọi là **shuffle**. Nó ghi ra đĩa, đi qua network, rồi đọc lại — đắt hơn
mọi thứ khác vài bậc.

In [16]:
def stage_cua_job_cuoi():
    jobs = sorted(api(f'/api/v1/applications/{APP}/jobs'), key=lambda j: j['jobId'])
    return jobs[-1]['jobId'], len(jobs[-1]['stageIds'])

def chay(dfx):
    # Sink `noop` = ghi vào hư không. Buộc Spark tính toàn bộ nhưng không tốn
    # một giây nào cho việc ghi đĩa — cách chuẩn để đo riêng phần TÍNH.
    # (`count()` không dùng được ở đây: bản thân nó đã kèm một shuffle.)
    dfx.write.format('noop').mode('overwrite').save()

chay(df.filter(F.col('trip_distance') > 1).select('PULocationID', 'total_amount'))
print('narrow (filter + select) → job {}, {} stage'.format(*stage_cua_job_cuoi()))

chay(df.groupBy('PULocationID').agg(F.avg('total_amount')))
print('wide   (groupBy + avg)   → job {}, {} stage'.format(*stage_cua_job_cuoi()))

narrow (filter + select) → job 9, 1 stage
wide   (groupBy + avg)   → job 11, 2 stage


### Ranh giới stage **chính là** shuffle

Job narrow có **1** stage. Job wide có **2**. Không phải trùng hợp:

> **Spark cắt job thành stage mới ở đúng chỗ có shuffle.**

Đó là quy tắc đọc Spark UI quan trọng nhất. Mở http://localhost:4040 → tab **Stages**,
bấm vào job wide vừa chạy, xem cột **Shuffle Read / Shuffle Write**. Con số ở đó là số
byte thật đã bò qua network.

Từ giờ, mỗi khi nhìn một job chậm, câu hỏi đầu tiên luôn là: *stage nào ghi shuffle
nhiều nhất, và có cách nào xoá nó đi không?*

In [17]:
# Nhìn shuffle bằng số, không bằng cảm giác.
# Xếp theo lượng shuffle ghi ra — stage cuối thường bé tí, không nói lên gì.
sts = sorted(api(f'/api/v1/applications/{APP}/stages'),
             key=lambda s: s.get('shuffleWriteBytes', 0), reverse=True)[:3]
print(f"{'stage':>6} {'task':>5} {'shuffle write':>15} {'shuffle read':>14}   {'mô tả'}")
for s in sts:
    print(f"{s['stageId']:>6} {s['numTasks']:>5} "
          f"{s.get('shuffleWriteBytes',0)/1e6:>12.1f} MB {s.get('shuffleReadBytes',0)/1e6:>11.1f} MB   "
          f"{s['name'][:44]}")

 stage  task   shuffle write   shuffle read   mô tả
    16     6          0.1 MB         0.0 MB   Spark Connect - session_id: "49b2cd02-39e1-4
     6     6          0.0 MB         0.0 MB   Spark Connect - session_id: "49b2cd02-39e1-4
     3     6          0.0 MB         0.0 MB   Spark Connect - session_id: "49b2cd02-39e1-4


---
## Bước 5 · SAI CÓ CHỦ ĐÍCH ① — viết một job chậm thảm hại, rồi tự chữa

Câu hỏi nghiệp vụ rất đơn giản: **quận nào (borough) khách boa hào phóng nhất?**

Ta sẽ viết nó theo cách *tự nhiên nhưng sai*, đo, rồi sửa từng lỗi một và đo lại sau
mỗi lần. Mục tiêu: **nhanh hơn ≥3 lần**.

In [18]:
# Bước 5-7 cố ý chỉ dùng 6 tháng thay vì 12. Lý do rất cụ thể:
# bản CHẬM kết hợp sort-merge join VỚI UDF Python, và trên Docker VM ~8GB thì
# đúng tổ hợp đó làm executor bị kernel giết (exit 137). Từng thứ RIÊNG LẺ thì
# 12 tháng vẫn chạy tốt — bạn sẽ thấy ở bước 6 và bước 8.
# Máy rộng RAM hơn: đổi SO_THANG = 12 rồi chạy lại, kết luận không đổi.
SO_THANG = 6
FILE_THANG = [f's3a://{BUCKET}/raw/yellow_taxi/yellow_tripdata_2024-{m:02d}.parquet'
              for m in range(1, SO_THANG + 1)]

trips = (spark.read.parquet(*FILE_THANG)
              .filter((F.col('total_amount') > 0) & (F.col('tip_amount') >= 0)))
print(f'{SO_THANG} tháng dữ liệu: {trips.count():,} chuyến')

zones = (spark.read.option('header', True).option('inferSchema', True).csv(ZONES))
print('Bảng tra vùng:', zones.count(), 'dòng —', zones.columns)
zones.show(3)

6 tháng dữ liệu: 20,072,258 chuyến
Bảng tra vùng: 265 dòng — ['LocationID', 'Borough', 'Zone', 'service_zone']
+----------+-------+--------------------+------------+
|LocationID|Borough|                Zone|service_zone|
+----------+-------+--------------------+------------+
|         1|    EWR|      Newark Airport|         EWR|
|         2| Queens|         Jamaica Bay|   Boro Zone|
|         3|  Bronx|Allerton/Pelham G...|   Boro Zone|
+----------+-------+--------------------+------------+
only showing top 3 rows


In [19]:
# ══ BẢN CHẬM ══ Ba lỗi cố ý, đều là lỗi người mới hay mắc.

# LỖI ①: tắt broadcast join. Bảng zones chỉ 265 dòng nhưng vẫn bị đem đi shuffle.
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)

# LỖI ②: UDF Python. Mỗi dòng phải rời JVM, sang tiến trình Python, rồi quay về.
@F.udf('double')
def tip_pct_udf(tip, total):
    if tip is None or total is None or total == 0:
        return None
    return float(tip) / float(total) * 100

cham = (trips
        .repartition(500)                       # LỖI ③: shuffle hoàn toàn vô nghĩa
        .join(zones, trips.PULocationID == zones.LocationID)
        .withColumn('tip_pct', tip_pct_udf('tip_amount', 'total_amount'))
        .groupBy('Borough')
        .agg(F.avg('tip_pct').alias('tip_pct_tb'), F.count('*').alias('so_chuyen'))
        .orderBy(F.desc('tip_pct_tb')))

t_cham, _ = bench('BẢN CHẬM (3 lỗi)', lambda: cham.collect())

BẢN CHẬM (3 lỗi)                               14.50s


In [20]:
cham.explain(mode='simple')

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [tip_pct_tb#134 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(tip_pct_tb#134 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=866]
      +- HashAggregate(keys=[Borough#100], functions=[avg(tip_pct#133), count(1)])
         +- Exchange hashpartitioning(Borough#100, 200), ENSURE_REQUIREMENTS, [plan_id=863]
            +- HashAggregate(keys=[Borough#100], functions=[partial_avg(tip_pct#133), partial_count(1)])
               +- Project [Borough#100, pythonUDF0#162 AS tip_pct#133]
                  +- BatchEvalPython [tip_pct_udf(tip_amount#70, total_amount#73)#131], [pythonUDF0#162]
                     +- Project [tip_amount#70, total_amount#73, Borough#100]
                        +- SortMergeJoin [PULocationID#64], [LocationID#99], Inner
                           :- Sort [PULocationID#64 ASC NULLS FIRST], false, 0
                           :  +- Exchange hashpartitioning(PULocationID#64, 200), ENSURE_REQ

Trong plan trên tìm chữ **`SortMergeJoin`** và **`BatchEvalPython`**. Hai chữ đó là hai
trong ba lỗi, hiện nguyên hình.

Giờ sửa từng lỗi, đo lại sau mỗi lần — để biết **lỗi nào thật sự đắt**, chứ không sửa mò.

In [21]:
# ── Sửa lỗi ③: bỏ repartition(500) thừa ──
b1 = (trips
      .join(zones, trips.PULocationID == zones.LocationID)
      .withColumn('tip_pct', tip_pct_udf('tip_amount', 'total_amount'))
      .groupBy('Borough')
      .agg(F.avg('tip_pct').alias('tip_pct_tb'), F.count('*').alias('so_chuyen'))
      .orderBy(F.desc('tip_pct_tb')))
t_b1, _ = bench('+ bỏ repartition(500)', lambda: b1.collect())

+ bỏ repartition(500)                           5.64s


In [22]:
# ── Sửa tiếp lỗi ①: cho phép broadcast bảng nhỏ ──
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', 10 * 1024 * 1024)

b2 = (trips
      .join(F.broadcast(zones), trips.PULocationID == zones.LocationID)
      .withColumn('tip_pct', tip_pct_udf('tip_amount', 'total_amount'))
      .groupBy('Borough')
      .agg(F.avg('tip_pct').alias('tip_pct_tb'), F.count('*').alias('so_chuyen'))
      .orderBy(F.desc('tip_pct_tb')))
t_b2, _ = bench('+ broadcast bảng zones', lambda: b2.collect())

+ broadcast bảng zones                          4.04s


In [23]:
# ── Sửa nốt lỗi ②: thay UDF Python bằng biểu thức gốc ──
nhanh = (trips
         .join(F.broadcast(zones), trips.PULocationID == zones.LocationID)
         .withColumn('tip_pct', F.col('tip_amount') / F.col('total_amount') * 100)
         .groupBy('Borough')
         .agg(F.avg('tip_pct').alias('tip_pct_tb'), F.count('*').alias('so_chuyen'))
         .orderBy(F.desc('tip_pct_tb')))
t_nhanh, kq = bench('+ bỏ UDF Python  ← BẢN NHANH', lambda: nhanh.collect())

print()
print(f'{"":<44}{"giây":>8}{"tăng tốc":>12}')
for nhan, t in [('bản chậm ban đầu', t_cham), ('bỏ repartition', t_b1),
                ('+ broadcast', t_b2), ('+ bỏ UDF', t_nhanh)]:
    print(f'{nhan:<44}{t:>8.2f}{t_cham/t:>11.1f}×')

+ bỏ UDF Python  ← BẢN NHANH                    0.92s

                                                giây    tăng tốc
bản chậm ban đầu                               14.50        1.0×
bỏ repartition                                  5.64        2.6×
+ broadcast                                     4.04        3.6×
+ bỏ UDF                                        0.92       15.7×


In [24]:
print('Quận nào boa hào phóng nhất:\n')
print(f"{'Borough':<16}{'tip %':>8}{'số chuyến':>14}")
for r in kq:
    if r['Borough'] and r['tip_pct_tb'] is not None:
        print(f"{r['Borough']:<16}{r['tip_pct_tb']:>8.2f}{r['so_chuyen']:>14,}")

Quận nào boa hào phóng nhất:

Borough            tip %     số chuyến
Unknown            11.88        64,899
Manhattan          11.48    17,901,138
Queens             10.51     1,782,821
EWR                 9.74         2,469
N/A                 6.91        10,740
Brooklyn            3.31       251,147
Staten Island       2.58           648
Bronx               0.62        58,396


### Ba lỗi, xếp theo mức độ đắt

**`repartition(500)` — shuffle mua vui.** Ép toàn bộ dữ liệu qua network để chia lại thành
500 mảnh, trong khi cluster chỉ có 4 core. 500 task chờ 4 chỗ ngồi, và mỗi task chỉ ôm một
mẩu tí xíu. Không đổi lấy được gì cả.

**Sort-merge join cho một bảng 265 dòng.** Không có broadcast, Spark shuffle *cả hai* phía
theo khoá join — nghĩa là ném cả bảng 36 triệu dòng qua network để ghép với một bảng bé
bằng cái danh bạ. Bước 6 sẽ mổ kỹ chỗ này.

**UDF Python — thuế phải trả ở mỗi dòng.** Đây thường là thủ phạm nặng nhất. Dữ liệu nằm
trong JVM ở dạng nhị phân đã nén. UDF Python bắt từng dòng phải: serialize → gửi sang một
tiến trình Python riêng → deserialize → chạy hàm → serialize → gửi về → deserialize.
Nhân với 36 triệu.

Tệ hơn nữa: **Catalyst không nhìn thấy bên trong UDF**. Với nó, `tip_pct_udf` là một hộp
đen — không tối ưu được, không đẩy xuống được, không sinh mã máy được. Còn
`F.col('tip_amount') / F.col('total_amount')` thì Catalyst hiểu, và biên dịch thẳng thành
bytecode JVM chạy trên dữ liệu cột.

> **Quy tắc:** trước khi viết một UDF Python, hãy chắc chắn `pyspark.sql.functions` thật sự
> không có hàm bạn cần. 95% trường hợp là có.

---
## Bước 6 · Broadcast join vs sort-merge join

Bước 5 gộp ba lỗi. Giờ tách riêng chuyện join ra đo cho rõ, vì đây là kỹ thuật tối ưu
được hỏi nhiều nhất trong phỏng vấn.

In [25]:
def dem_join(dung_broadcast):
    z = F.broadcast(zones) if dung_broadcast else zones
    return (trips.join(z, trips.PULocationID == zones.LocationID)
                 .groupBy('Borough').agg(F.count('*').alias('n')))

spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
t_smj, _ = bench('sort-merge join (shuffle cả hai phía)', lambda: dem_join(False).collect())

spark.conf.set('spark.sql.autoBroadcastJoinThreshold', 10 * 1024 * 1024)
t_bhj, _ = bench('broadcast join   (không shuffle)', lambda: dem_join(True).collect())

print(f'\nBroadcast nhanh hơn {t_smj/t_bhj:.1f}×')

sort-merge join (shuffle cả hai phía)           2.65s
broadcast join   (không shuffle)                0.85s

Broadcast nhanh hơn 3.1×


In [26]:
print('══ KHÔNG broadcast ' + '═'*40)
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
dem_join(False).explain(mode='simple')

print('\n══ CÓ broadcast ' + '═'*43)
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', 10 * 1024 * 1024)
dem_join(True).explain(mode='simple')

══ KHÔNG broadcast ════════════════════════════════════════
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Borough#100], functions=[count(1)])
   +- Exchange hashpartitioning(Borough#100, 200), ENSURE_REQUIREMENTS, [plan_id=2030]
      +- HashAggregate(keys=[Borough#100], functions=[partial_count(1)])
         +- Project [Borough#100]
            +- SortMergeJoin [PULocationID#64], [LocationID#99], Inner
               :- Sort [PULocationID#64 ASC NULLS FIRST], false, 0
               :  +- Exchange hashpartitioning(PULocationID#64, 200), ENSURE_REQUIREMENTS, [plan_id=2022]
               :     +- Project [PULocationID#64]
               :        +- Filter ((((isnotnull(total_amount#73) AND isnotnull(tip_amount#70)) AND (total_amount#73 > 0.0)) AND (tip_amount#70 >= 0.0)) AND isnotnull(PULocationID#64))
               :           +- FileScan parquet [PULocationID#64,tip_amount#70,total_amount#73] Batched: true, DataFilters: [isnotnull(total_amount#73), i

### Đọc hai plan trên

Bản không broadcast: `SortMergeJoin`, và **hai** node `Exchange hashpartitioning` — hai
shuffle, một cho mỗi phía. Bảng 36 triệu dòng phải sắp xếp và bò qua network.

Bản có broadcast: `BroadcastHashJoin`, và **`Exchange` biến mất** ở phía bảng lớn.

Cơ chế: bảng nhỏ được driver gom về, gửi **một bản sao đầy đủ tới từng executor**. Sau đó
mỗi executor tra bảng ngay trên dữ liệu sẵn có của mình. Bảng lớn **không nhúc nhích**.

Broadcast không phải "nhanh hơn một chút" — nó **xoá bỏ hẳn một shuffle**. Đó là khác biệt
về bậc, không phải về mức độ.

**Giới hạn:** bảng nhỏ phải chui vừa RAM của *mỗi* executor, và phải đi qua driver trước.
Mặc định Spark tự broadcast bảng dưới 10MB (`autoBroadcastJoinThreshold`). Broadcast một
bảng 2GB là cách chắc chắn nhất để nhận `OutOfMemoryError` ở driver.

---
## Bước 7 · SAI CÓ CHỦ ĐÍCH ② — data skew

Tình huống kinh điển, và là câu phỏng vấn gặp thường xuyên:

> *Một stage có 200 task. 199 task xong trong 2 giây. Task cuối chạy 4 phút.*

Cluster 4 core nhưng chỉ 1 core làm việc, 3 core ngồi nhìn. Đó là **data skew**: một khoá
ôm quá nhiều dữ liệu so với phần còn lại.

Ta tạo ra nó có chủ đích: giả sử 90% chuyến xe đều đón ở cùng một điểm.

In [27]:
# AQE tắt trước, để nhìn thấy skew ở dạng thô. Bật lại ở cuối bước.
spark.conf.set('spark.sql.adaptive.enabled', False)

lech = trips.withColumn(
    'diem_don',
    F.when(F.rand(42) < 0.9, F.lit(132)).otherwise(F.col('PULocationID'))  # 132 = JFK
)

print('Phân bố khoá — 5 khoá đông nhất:')
lech.groupBy('diem_don').count().orderBy(F.desc('count')).show(5)

Phân bố khoá — 5 khoá đông nhất:
+--------+--------+
|diem_don|   count|
+--------+--------+
|     132|18152710|
|     161|   94082|
|     237|   93719|
|     236|   87209|
|     162|   69681|
+--------+--------+
only showing top 5 rows


In [28]:
def phan_bo_task():
    '''Phân bố thời gian task của stage nặng nhất TRONG JOB VỪA CHẠY.

    Phải bám vào job cuối cùng, không phải toàn bộ application: nếu chọn
    "stage shuffle lớn nhất từ trước tới nay" thì lần đo nào cũng ra cùng
    một stage cũ, và ba phép so sánh bên dưới sẽ giống hệt nhau — một cái
    bẫy rất dễ mắc khi đọc REST API của Spark.
    '''
    job = sorted(api(f'/api/v1/applications/{APP}/jobs'), key=lambda j: j['jobId'])[-1]
    cua_job = set(job['stageIds'])
    sts = [s for s in api(f'/api/v1/applications/{APP}/stages')
           if s['stageId'] in cua_job and s['status'] == 'COMPLETE']
    # Stage nào ĐỌC shuffle nhiều nhất = stage rút gọn, nơi skew lộ ra
    st = max(sts, key=lambda s: s.get('shuffleReadBytes', 0))
    d = api(f"/api/v1/applications/{APP}/stages/{st['stageId']}/{st['attemptId']}"
            f"/taskSummary?quantiles=0,0.25,0.5,0.75,0.95,1.0")
    q = d['executorRunTime']
    print(f"job {job['jobId']} · stage {st['stageId']} — {st['numTasks']} task, "
          f"thời gian chạy (ms):")
    for ten, v in zip(['min', '25%', 'trung vị', '75%', '95%', 'MAX'], q):
        print(f'  {ten:<10}{v:>10,.0f}')
    print(f'  → task chậm nhất gấp {q[-1]/max(q[2],1):.0f}× trung vị')
    return q

# Chạm vào dữ liệu một lần trước khi đo, để phí liệt kê file không lẫn vào con số
_ = lech.count()

t_gom, _ = bench('groupBy + avg trên khoá LỆCH',
                 lambda: lech.groupBy('diem_don').agg(F.avg('total_amount')).collect())
print()
q_gom = phan_bo_task()

groupBy + avg trên khoá LỆCH                    0.75s

job 52 · stage 107 — 200 task, thời gian chạy (ms):
  min                0
  25%                1
  trung vị           1
  75%                2
  95%                4
  MAX               18
  → task chậm nhất gấp 18× trung vị


### Bất ngờ thứ nhất: `groupBy` **không** hề bị skew

90% dữ liệu dồn vào một khoá, mà task chậm nhất chỉ nhỉnh hơn trung vị vài chục mili-giây.
Vì sao?

Vì **partial aggregation**. Trước khi shuffle, mỗi partition đã tự gom nhóm cục bộ rồi —
mười triệu dòng khoá `132` co lại thành **một** dòng `(132, tổng, đếm)`. Thứ thật sự bò qua
network chỉ là vài trăm dòng. Khoá có lệch đến đâu cũng không quan trọng.

> **Bài học ngược với trực giác:** không phải cứ khoá lệch là chậm. Skew chỉ đau khi
> phép tính **không gộp trước được** — điển hình nhất là **join**.

Thử lại bằng join, tắt broadcast để buộc Spark shuffle cả phía bảng lớn:

In [29]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)   # ép sort-merge join

t_join, _ = bench('JOIN trên khoá lệch (AQE tắt, không broadcast)', lambda: (
    lech.join(zones, lech.diem_don == zones.LocationID)
        .groupBy('Borough').agg(F.count('*'))
        .collect()))
print()
q_join = phan_bo_task()

JOIN trên khoá lệch (AQE tắt, không broadcast)    5.10s

job 53 · stage 110 — 200 task, thời gian chạy (ms):
  min                1
  25%                2
  trung vị           4
  75%                7
  95%               20
  MAX            3,729
  → task chậm nhất gấp 932× trung vị


### Đây mới là skew thật

Lần này chênh lệch giữa **trung vị** và **MAX** giãn hẳn ra. Join không gộp trước được:
mọi dòng có `diem_don = 132` bắt buộc phải nằm cùng một partition để gặp dòng tương ứng
bên bảng kia — mà một partition thì chỉ một task xử lý, chỉ một core chạy.

Mở :4040 → Stages → stage vừa chạy → **Event Timeline**: một thanh dài chạy mãi trong khi
các thanh khác đã kết thúc từ lâu.

### Cách chữa ①: salting — bẻ khoá to thành nhiều khoá nhỏ

Gắn thêm một số ngẫu nhiên `0..N-1` vào khoá bên bảng lớn, và **nhân bản** bảng nhỏ ra
đủ `N` bản, mỗi bản mang một giá trị muối. Khoá `132` vỡ thành `N` khoá con, rải đều ra
`N` partition. Kết quả join không đổi, nhưng việc được chia cho nhiều core.

In [30]:
SALT = 16

# Bảng lớn: thêm cột muối ngẫu nhiên
lech_muoi = lech.withColumn('muoi', (F.rand(7) * SALT).cast('int'))

# Bảng nhỏ: nhân bản ra 16 bản, mỗi bản một giá trị muối
zones_muoi = (zones
              .withColumn('muoi', F.explode(F.array(*[F.lit(k) for k in range(SALT)]))))

t_muoi, _ = bench('JOIN có salting (AQE vẫn tắt)', lambda: (
    lech_muoi.join(zones_muoi,
                   (lech_muoi.diem_don == zones_muoi.LocationID) &
                   (lech_muoi.muoi == zones_muoi.muoi))
             .groupBy('Borough').agg(F.count('*'))
             .collect()))
print()
q_muoi = phan_bo_task()

JOIN có salting (AQE vẫn tắt)                   2.71s

job 54 · stage 114 — 200 task, thời gian chạy (ms):
  min                3
  25%                4
  trung vị           6
  75%                8
  95%              195
  MAX              228
  → task chậm nhất gấp 38× trung vị


### Cách chữa ②: để Spark tự lo — AQE

**AQE (Adaptive Query Execution)** là tính năng Spark 3+, bật mặc định từ 3.2. Nó xem
**thống kê thật của shuffle vừa ghi ra** rồi sửa kế hoạch giữa chừng: gộp partition bé, và
**tự cắt nhỏ partition lệch** (`skewJoin`) — đúng việc salting làm, nhưng không cần sửa code.

Nãy giờ ta tắt nó đi để nhìn thấy vấn đề. Giờ bật lại:

In [31]:
spark.conf.set('spark.sql.adaptive.enabled', True)
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', True)

t_aqe, _ = bench('JOIN trên khoá lệch (AQE BẬT, code không đổi)', lambda: (
    lech.join(zones, lech.diem_don == zones.LocationID)
        .groupBy('Borough').agg(F.count('*'))
        .collect()))

spark.conf.set('spark.sql.autoBroadcastJoinThreshold', 10 * 1024 * 1024)  # trả lại mặc định

print()
print(f'{"":<46}{"giây":>8}')
for nhan, t in [('groupBy + avg (miễn nhiễm skew)', t_gom),
                ('JOIN khoá lệch, AQE tắt', t_join),
                ('JOIN + salting thủ công', t_muoi),
                ('JOIN, AQE bật, không sửa code', t_aqe)]:
    print(f'{nhan:<46}{t:>8.2f}')

JOIN trên khoá lệch (AQE BẬT, code không đổi)    5.04s

                                                  giây
groupBy + avg (miễn nhiễm skew)                   0.75
JOIN khoá lệch, AQE tắt                           5.10
JOIN + salting thủ công                           2.71
JOIN, AQE bật, không sửa code                     5.04


### Kết luận thực dụng về skew

AQE xử lý phần lớn skew mà **không cần sửa một dòng code nào** — hãy để nó bật, đó là mặc
định vì lý do chính đáng.

Nhưng vẫn phải hiểu salting, vì hai lẽ: (1) AQE dựa vào thống kê sau shuffle, nên nó chỉ
*giảm nhẹ* chứ không xoá được skew cực đoan; (2) đây là câu phỏng vấn kinh điển, và trả lời
*"em bật AQE"* mà không giải thích được nó làm gì thì không qua được vòng đào sâu.

> **Cách nhận diện skew trong đời thực:** stage bị treo, Spark UI → Stages → cột
> **Duration**: `Max` lớn hơn `Median` vài chục lần. Đó là dấu hiệu, không phải "cluster yếu".

---
## Bước 8 · Spark mở đúng bảng Delta mà Phase 2 đã tạo

Ở Phase 2, bảng `phase2/bronze_trips` được ghi bằng **`delta-rs`** — thư viện Rust, không
có JVM, không biết Spark là gì.

Giờ Spark mở nó. Không export, không convert, không import. **Trỏ vào đúng đường dẫn cũ.**

In [32]:
d = spark.read.format('delta').load(DELTA_PHASE2)
print(f'Bảng Phase 2 đọc bằng Spark: {d.count():,} dòng, {len(d.columns)} cột')
d.select(d.columns[:5]).show(3)

Bảng Phase 2 đọc bằng Spark: 600,003 dòng, 8 cột
+-------+-------------------+-------+-------+---------------+
|trip_id|          pickup_at|pu_zone|do_zone|passenger_count|
+-------+-------------------+-------+-------+---------------+
|1000001|2024-01-06 16:38:19|    186|     79|              1|
|1000002|2024-01-06 16:38:19|    114|    230|              1|
|1000003|2024-01-06 16:38:20|    186|    239|              1|
+-------+-------------------+-------+-------+---------------+
only showing top 3 rows


In [33]:
# Lịch sử giao dịch — chính những file JSON bạn đã mở bằng tay ở Phase 2 bước 2,
# nay Spark đọc lại và trình bày dưới dạng bảng.
spark.sql(f"DESCRIBE HISTORY delta.`{DELTA_PHASE2}`") \
     .select('version', 'timestamp', 'operation', 'operationMetrics') \
     .show(20, truncate=60)

+-------+-------------------+---------+------------------------------------------------------------+
|version|          timestamp|operation|                                            operationMetrics|
+-------+-------------------+---------+------------------------------------------------------------+
|      5|2026-08-23 11:21:42|    WRITE|{num_partitions -> 0, execution_time_ms -> 4, num_added_r...|
|      4|2026-08-23 11:15:12|    MERGE|{num_target_files_skipped_during_scan -> 0, num_target_ro...|
|      3|2026-08-23 11:05:19|  RESTORE|                 {numRemovedFile -> 1, numRestoredFile -> 0}|
|      2|2026-08-23 11:01:01|    WRITE|{num_partitions -> 0, execution_time_ms -> 3, num_added_r...|
|      1|2026-08-23 10:56:41|    WRITE|{num_partitions -> 0, execution_time_ms -> 34, num_added_...|
|      0|2026-08-23 10:34:03|    WRITE|{num_partitions -> 0, execution_time_ms -> 98, num_added_...|
+-------+-------------------+---------+----------------------------------------------------

Cột `operation` ghi `WRITE`, `MERGE`, `OPTIMIZE`, `VACUUM`, `RESTORE`... — **do `delta-rs`
viết vào, và Spark đọc hiểu trọn vẹn**.

Đây là bằng chứng cụ thể cho luận điểm ở Phần 5 roadmap: **Delta là một *đặc tả*, không phải
một sản phẩm**. Bất kỳ engine nào cài đặt đúng đặc tả đó đều đọc/ghi được cùng một bảng.
So sánh với một warehouse đóng: muốn đưa dữ liệu ra ngoài thì phải `EXPORT`.

Giờ chiều ngược lại — Spark ghi, delta-rs đọc:

In [34]:
(nhanh.write.format('delta').mode('overwrite').save(GOLD))
print('✓ Spark đã ghi bảng Delta:', GOLD)

for k, sz in sorted((o['Key'], o['Size']) for o in
                    s3.list_objects_v2(Bucket=BUCKET, Prefix='phase3/gold_borough')['Contents']):
    print(f'{sz/1024:9.1f} KB  {k}')

✓ Spark đã ghi bảng Delta: s3a://lakehouse/phase3/gold_borough
      2.5 KB  phase3/gold_borough/_delta_log/00000000000000000000.crc
      1.3 KB  phase3/gold_borough/_delta_log/00000000000000000000.json
      0.0 KB  phase3/gold_borough/_delta_log/_staged_commits/
      1.2 KB  phase3/gold_borough/part-00000-425ef98a-df79-41db-875f-51e75e91005f-c000.snappy.parquet


In [35]:
# Đọc lại bằng delta-rs — thư viện KHÔNG hề biết Spark tồn tại
from deltalake import DeltaTable

SO = {
    'AWS_ACCESS_KEY_ID'     : os.environ['AWS_ACCESS_KEY_ID'],
    'AWS_SECRET_ACCESS_KEY' : os.environ['AWS_SECRET_ACCESS_KEY'],
    'AWS_ENDPOINT_URL'      : os.environ['AWS_ENDPOINT_URL'],
    'AWS_REGION'            : 'us-east-1',
    'AWS_ALLOW_HTTP'        : 'true',
}
dt = DeltaTable(f's3://{BUCKET}/phase3/gold_borough', storage_options=SO)
print(f'delta-rs đọc bảng do Spark ghi — version {dt.version()}:\n')
print(dt.to_pandas().to_string(index=False))

delta-rs đọc bảng do Spark ghi — version 0:

      Borough  tip_pct_tb  so_chuyen
      Unknown   11.877185      64899
    Manhattan   11.477642   17901138
       Queens   10.510707    1782821
          EWR    9.739690       2469
          N/A    6.914969      10740
     Brooklyn    3.307796     251147
Staten Island    2.578336        648
        Bronx    0.621999      58396


Một bảng, hai engine hoàn toàn khác nhau (JVM ↔ Rust), đọc và ghi qua lại tự nhiên.
**Đó là ý nghĩa thật của "định dạng mở".**

---
## Bước 9 · Khi nào **không** nên dùng Spark

Bước cuối này quan trọng hơn vẻ ngoài của nó. Biết giới hạn của một công cụ là dấu hiệu
đã dùng nó thật, chứ không chỉ đọc về nó.

Cùng câu hỏi nghiệp vụ của bước 5, chạy bằng DuckDB một máy — công cụ của Phase 0.

In [36]:
import duckdb, glob

con = duckdb.connect()
# CÙNG 6 tháng Spark đã dùng — so sánh khác lượng dữ liệu là so sánh gian lận
files = [f'/home/jovyan/data/yellow_tripdata_2024-{m:02d}.parquet'
         for m in range(1, SO_THANG + 1)]
print(f'{len(files)} file cục bộ, {sum(os.path.getsize(f) for f in files)/1e6:.0f} MB\n')

sql = f'''
    SELECT z.Borough,
           avg(t.tip_amount / t.total_amount * 100) AS tip_pct_tb,
           count(*)                                 AS so_chuyen
    FROM read_parquet({files!r}) t
    JOIN read_csv('/home/jovyan/data/taxi_zone_lookup.csv') z
      ON t.PULocationID = z.LocationID
    WHERE t.total_amount > 0 AND t.tip_amount >= 0
    GROUP BY z.Borough
    ORDER BY tip_pct_tb DESC
'''
t_duck, kq_duck = bench('DuckDB — MỘT tiến trình, không cluster', lambda: con.sql(sql).df())
print()
print(kq_duck.to_string(index=False))

6 file cục bộ, 342 MB

DuckDB — MỘT tiến trình, không cluster          0.24s

      Borough  tip_pct_tb  so_chuyen
      Unknown   11.877185      64899
    Manhattan   11.477642   17901138
       Queens   10.510707    1782821
          EWR    9.739690       2469
          N/A    6.914969      10740
     Brooklyn    3.307796     251147
Staten Island    2.578336        648
        Bronx    0.621999      58396


In [37]:
print(f'{"":<44}{"giây":>8}')
print(f'{"Spark, 2 executor, bản đã tối ưu":<44}{t_nhanh:>8.2f}')
print(f'{"Spark, bản chậm ban đầu":<44}{t_cham:>8.2f}')
print(f'{"DuckDB, một tiến trình":<44}{t_duck:>8.2f}')

                                                giây
Spark, 2 executor, bản đã tối ưu                0.92
Spark, bản chậm ban đầu                        14.50
DuckDB, một tiến trình                          0.24


### Vì sao DuckDB thắng ở quy mô này

**Spark trả phí cố định cho mọi job**, dù dữ liệu to hay bé: lập kế hoạch, tuần tự hoá
task, gửi qua network, chờ executor, gom kết quả. Vài trăm mili-giây đến vài giây, không
phụ thuộc kích thước dữ liệu.

Vài chục triệu dòng nghe to, nhưng vài trăm MB Parquet thì **vừa RAM một máy**. DuckDB đọc
thẳng, không phải nói chuyện với ai, không shuffle qua network.

**Đường cong đổi chiều ở đâu:**

| Quy mô | Nên dùng |
|---|---|
| < 10 GB | DuckDB / Polars / pandas — Spark chỉ tổ phiền |
| 10–200 GB | Ranh giới. Một máy to thường vẫn thắng |
| > vài trăm GB, hoặc phải chạy 24/7 có phục hồi lỗi | Spark |

Và chú ý một điều: chính cái trần RAM ~8GB đã buộc bước 5 phải rút xuống 6 tháng. Một
cluster 4 core / 6GB **không phải** là "dữ liệu lớn" — đó cũng là một dữ kiện của bài học này.

**Vậy học Spark để làm gì?** Vì tin tuyển dụng hỏi nó, đúng, nhưng còn vì thứ Spark dạy:
shuffle, partition, skew, lazy plan, chia việc trên nhiều máy — những khái niệm **không
thuộc riêng Spark**. Trino, BigQuery, Snowflake, Flink đều xoay quanh đúng chừng đó ý niệm.

Và câu trả lời phỏng vấn tốt nhất cho *"khi nào bạn dùng Spark?"* không bao giờ là
*"lúc nào cũng dùng"*.

---
## Tổng kết Phase 3

Bạn vừa dựng xong tầng 3 và có đủ ba mảnh cốt lõi của một lakehouse:

| Tầng | Ta dùng | Đã xong ở |
|---|---|---|
| 1 · Object storage | MinIO | Phase 0 |
| 2 · Table format | Delta Lake | Phase 2 |
| 3 · Compute engine | **Apache Spark** | **Phase 3** |

### Năm câu tự kiểm tra

Trả lời **bằng lời của mình**, dựa vào số bạn vừa đo — không chép lại từ markdown:

1. Driver và executor, cái nào chạy code trong cell notebook này? Cái nào giữ kết quả `collect()`?
2. Vì sao `df.filter(...)` chạy tức thì, còn `df.filter(...).agg(avg(...))` thì không?
   Và vì sao `count()` lại nhanh hơn `avg()` nhiều lần trên cùng dữ liệu?
3. Shuffle tốn ở chỗ nào — đĩa, network, hay CPU? Kể **hai** cách xoá bỏ một shuffle.
4. Một stage có 200 task, 199 task xong trong 2 giây, 1 task chạy 4 phút. Chuyện gì đang
   xảy ra, và bạn sửa thế nào?
5. Dữ liệu 5GB, một máy 32GB RAM — vì sao Spark có thể **chậm hơn** DuckDB?

### Còn thiếu gì

Mọi logic biến đổi dữ liệu trong notebook này nằm rải rác trong các cell. Không ai chạy lại
được, không ai test được, không ai biết bảng nào phụ thuộc bảng nào.

Đó là vấn đề **Phase 4 — Medallion + dbt** giải quyết.

In [38]:
spark.stop()
print('✓ Đã ngắt kết nối. Cluster vẫn chạy — `make down` khi muốn tắt hẳn.')

✓ Đã ngắt kết nối. Cluster vẫn chạy — `make down` khi muốn tắt hẳn.
